# Notebook 04 — Baseline Models and Ridge Regression

**Synopsis Objective 4, part 1 of 2: baseline and linear regression models.** (Part 2 — gradient boosting with LightGBM — is notebook `05_gradient_boosting_model`.)

Before committing to the full 5-fold walk-forward evaluation protocol in notebook `06_walkforward_validation_and_evaluation`, I want to sanity-check that three things actually work end-to-end on this dataset:

1. The two baseline predictors (`seasonal_naive_predict`, `moving_average_predict`) read the right pre-computed feature columns and produce sensible-looking Sales predictions.
2. The `build_ridge_pipeline` estimator from `src/models.py` trains on the log-sales target and the shared `NUMERIC_FEATURE_COLS` / `CATEGORICAL_FEATURE_COLS` contract without blowing up — no column mismatches, leftover NaNs, or dtype surprises.
3. The metrics pipeline in `src/metrics.py` — in particular the per-store-scaled MASE — runs cleanly against all three sets of predictions.

To check this I carve out **one illustrative 42-day holdout** — the same length as the real Kaggle competition's test horizon — using `walk_forward_splits(..., n_folds=1, val_days=42)`. That gives a single, chronologically clean train/validation split, rather than the 5 expanding-window folds notebook 06 will use.

**This is explicitly a plumbing check, not a model comparison.** Whichever model happens to score best on this one 42-day window below is *not* the project's final answer for which model to adopt — a single holdout window is too easily won by luck (how holidays or promo cycles happened to fall in that window) to support that conclusion. The real comparison, averaged across 5 non-overlapping expanding-window folds, happens in notebook 06.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## Imports

Everything below comes straight from the tested `src/` package — no modeling logic is written in this notebook, only orchestration. `NUMERIC_FEATURE_COLS` and `CATEGORICAL_FEATURE_COLS` are the shared modeling contract defined once in `src/features.py` and reused unchanged by every notebook from 04 through 08, so the baselines, Ridge here, and LightGBM in notebook 05 are always compared on exactly the same feature set.

In [ ]:
import numpy as np
from src.data_io import load_parquet
from src.features import NUMERIC_FEATURE_COLS, CATEGORICAL_FEATURE_COLS
from src.models import seasonal_naive_predict, moving_average_predict, build_ridge_pipeline
from src.validation import walk_forward_splits
from src.metrics import evaluate_all

## Load the engineered feature matrix

`train_features.parquet` is the artifact produced by notebook 03: the calendar/lag/rolling feature set from `build_feature_matrix`, already passed through `filter_trainable_rows` (so `Open == 0` rows and the opening ~28 days of each store's history, where the longest lag/rolling window hasn't filled in yet, are already dropped) and already checked with `assert_no_leakage`. I re-sort by `(Store, Date)` and reset the index anyway — it costs nothing and guards against ever accidentally feeding an unsorted frame into the walk-forward splitter or a per-store lag computation downstream.

In [ ]:
df = load_parquet("../data/processed/train_features.parquet").sort_values(["Store", "Date"]).reset_index(drop=True)
FEATURE_COLS = NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS

A couple of quick checks before splitting anything: the feature columns should have zero missing values (confirming `filter_trainable_rows` did its job upstream) and the log-target should be fully populated.

In [ ]:
print(f"Rows: {len(df):,} | Stores: {df['Store'].nunique():,} | Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
assert df[FEATURE_COLS].isna().sum().sum() == 0, "Feature columns should already be fully populated -- train_features.parquet is expected to be the trainable subset saved by notebook 03"
assert df["log_sales"].isna().sum() == 0

## One illustrative train/validation split

`walk_forward_splits` is the same expanding-window splitter notebook 06 will call with `n_folds=5`. Here it's called with `n_folds=1` and `val_days=42` to get exactly one fold back: the last 42 days of the dataset held out as validation, everything strictly before that as training. The explicit assertion that `train_df["Date"].max() < val_df["Date"].min()` is a belt-and-suspenders leakage guard — it should already be structurally guaranteed by the splitter, but it costs one line to re-verify rather than trust it silently.

In [ ]:
split = walk_forward_splits(df["Date"], n_folds=1, val_days=42)[0]
train_df = df.loc[split["train_mask"]]
val_df = df.loc[split["val_mask"]]
assert train_df["Date"].max() < val_df["Date"].min()

Quick look at what the split actually produced, in terms of dates, row counts, and store coverage:

In [ ]:
print(f"Train: {train_df['Date'].min().date()} to {train_df['Date'].max().date()} ({len(train_df):,} rows, {train_df['Store'].nunique():,} stores)")
print(f"Val:   {val_df['Date'].min().date()} to {val_df['Date'].max().date()} ({len(val_df):,} rows, {val_df['Store'].nunique():,} stores)")
print(f"Val window length: {(val_df['Date'].max() - val_df['Date'].min()).days + 1} days")

## Baseline predictions

Both baselines are zero-training — they just read a pre-computed column off the validation rows:

- **`seasonal_naive_predict`** reads `lag_7`: "this store will sell today what it sold on the same day of the week, one week ago." This is the natural baseline for retail sales, which has a strong weekly (day-of-week / promo-cycle) seasonal pattern.
- **`moving_average_predict`** reads `roll_mean_7`: the trailing 7-day average, computed with a 1-day shift so it never looks at the day being predicted.

Any model that earns a place in this pipeline — Ridge here, LightGBM in notebook 05 — has to beat these on held-out data. Failing to beat a naive lag-7 baseline would mean the model isn't adding any value over the plain week-over-week sales pattern.

In [ ]:
naive_pred = seasonal_naive_predict(val_df)
ma_pred = moving_average_predict(val_df)

## Ridge regression pipeline

`build_ridge_pipeline` wires up an sklearn `Pipeline`: `StandardScaler` on the numeric columns, `OneHotEncoder(handle_unknown="ignore")` on the categorical columns — including `Store` itself, so every store gets its own fixed-effect-style indicator on top of its numeric covariates — feeding a plain `Ridge(alpha=1.0)`. It's trained on `log_sales` (`log1p(Sales)`) rather than raw `Sales`, because Sales is heavily right-skewed across roughly 1,115 stores of very different sizes, and log-space keeps the squared-error loss from being dominated by the handful of largest stores. Predictions therefore need `np.expm1()` to come back onto the Sales scale before they're comparable to the baselines or to actual Sales.

In [ ]:
ridge = build_ridge_pipeline(NUMERIC_FEATURE_COLS, CATEGORICAL_FEATURE_COLS)
ridge.fit(train_df[FEATURE_COLS], train_df["log_sales"])
ridge_pred = np.expm1(ridge.predict(val_df[FEATURE_COLS]))

## Comparing all three on the holdout

`evaluate_all` bundles RMSE, MAE, WMAPE, and MASE. The one argument worth calling out is `train_groups=train_df["Store"]` passed into MASE: this is a *global* model trained across every store at once, and the MASE scale is a naive (lag-7) error computed as a positional diff over the training target. Without `train_groups`, that diff would be taken over the concatenated, `(Store, Date)`-sorted training array and would occasionally subtract one store's Sales from a completely different store's Sales right at a store boundary — a meaningless number. Passing `train_groups` makes `mase()` compute that scale within each store's own series and pool the result, which is the only version of MASE that's valid for a multi-store model.

In [ ]:
import pandas as pd

rows = []
for name, pred in [("seasonal_naive", naive_pred), ("moving_average", ma_pred), ("ridge", ridge_pred)]:
    m = evaluate_all(val_df["Sales"], pred, train_df["Sales"], seasonal_period=7, train_groups=train_df["Store"])
    rows.append({"model": name, **m})
comparison = pd.DataFrame(rows)
comparison

## Predicted vs. actual: a visual gut check

This is not a rigorous residual analysis — that belongs with whichever model survives the formal evaluation. It's a quick visual check that Ridge's predictions land in the right ballpark across the whole range of Sales values on this holdout: roughly centered on the `y = x` line, without an obvious systematic bias (for example, consistently under-predicting the highest-volume days) that would point to something broken in the feature pipeline or the fit.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 6.5))

actual = val_df["Sales"].to_numpy()
upper = max(actual.max(), ridge_pred.max()) * 1.05

ax.scatter(actual, ridge_pred, s=8, alpha=0.12, color="#2a78d6", edgecolors="none", label="Ridge prediction")
ax.plot([0, upper], [0, upper], color="#52514e", linestyle="--", linewidth=1.5, label="y = x (perfect prediction)")

ax.set_xlim(0, upper)
ax.set_ylim(0, upper)
ax.set_xlabel("Actual Sales")
ax.set_ylabel("Predicted Sales")
ax.set_title("Ridge: Predicted vs Actual (illustrative single split)")
ax.grid(True, color="#e1e0d9", linewidth=0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="upper left", frameon=False)

plt.tight_layout()
plt.show()

## Takeaway

All three predictors ran end-to-end on this 42-day holdout without errors, and the metrics pipeline — including the per-store MASE scaling — computed cleanly for all three. That's all this notebook set out to confirm.

I'm deliberately not drawing a conclusion here about which model is "best": one 42-day window is one realization of whatever happened to be going on in that particular stretch of store traffic, promos, and holidays, and either baseline or Ridge could look good or bad here by chance alone. Gradient boosting with LightGBM is built out next, in notebook `05_gradient_boosting_model`, and then all four candidates — seasonal naive, moving average, Ridge, and LightGBM — get compared properly across the 5 expanding-window folds from `walk_forward_splits(..., n_folds=5, val_days=42)` in notebook `06_walkforward_validation_and_evaluation`. That averaged, multi-fold comparison is the one that actually decides the project's model choice.